# MeatGuard II — 04 · Evaluation of the explanations

`src/evaluate_xai.py` on the pruned, fine-tuned MeatGuard-S:

1. Grad-CAM layer chosen on a validation sample (largest mean insertion − deletion AUC).
2. On a session-stratified test sample (100 images per class, at most 2 per session): Grad-CAM, occlusion and SHAP
   (SLIC superpixels, 50 training images as background) scored by deletion/insertion AUC against a random pixel
   order and by the **meat share** of positive attribution against the mask area (needs `masks/` in the data zip).
3. Cascading weight-randomisation sanity check on 20 test images.

**Saved to `runs/xai/`:** `layer_selection.json`, `xai_per_image.csv`, `sanity_check.csv`, `xai_summary.json`,
`example_maps.npz`, `xai_examples.png`. The SHAP step is the slow one (roughly 1–3 s per image on a GPU).

## 1. Setup
Upload the whole `MeatGuard_II` folder to Google Drive and the data zip (`crops/`, `masks/`, `sessions.csv`) next to
it, then edit the two paths below if yours differ. Results are written to `MeatGuard_II/runs/` on Drive, so they
survive disconnects; re-running a training cell resumes from the last saved epoch.

In [ ]:
import os, sys
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")

# --- paths: edit these -------------------------------------------------------------------------
REPO_DIR = "/content/drive/MyDrive/MeatGuard_II" if IN_COLAB else os.path.abspath("..")   # the codebase on Drive
DATA_SRC = "/content/drive/MyDrive/MeatGuard/full.zip" if IN_COLAB else "E:/Datasets/full"  # zip or folder: crops/, masks/, sessions.csv
# -------------------------------------------------------------------------------------------------
DATA_ROOT = "/content/data/full" if IN_COLAB else DATA_SRC     # local copy on the Colab VM (Drive is slow for many small files)
RUNS = os.path.join(REPO_DIR, "runs")
WORKERS = 4 if IN_COLAB else 0
os.chdir(REPO_DIR)
sys.path.insert(0, REPO_DIR)
if IN_COLAB:
    !pip -q install torch-pruning shap
!nvidia-smi --query-gpu=name,memory.total --format=csv || echo "No GPU visible"
import torch, torchvision
print("torch", torch.__version__, "| torchvision", torchvision.__version__, "| CUDA", torch.cuda.is_available())

## 2. Data

In [ ]:
# Copy the data to the VM once, then check it: every session in one partition, every listed image present
import zipfile, shutil
import pandas as pd
from pathlib import Path
if IN_COLAB and not os.path.exists(os.path.join(DATA_ROOT, "sessions.csv")):
    if DATA_SRC.endswith(".zip"):
        with zipfile.ZipFile(DATA_SRC) as z:
            z.extractall(DATA_ROOT)
    else:
        shutil.copytree(DATA_SRC, DATA_ROOT)
if not os.path.exists(os.path.join(DATA_ROOT, "sessions.csv")):      # zip made from the parent folder
    nested = [p.parent for p in Path(DATA_ROOT).glob("*/sessions.csv")]
    DATA_ROOT = str(nested[0]) if nested else DATA_ROOT
CROPS, SPLIT, MASKS = (os.path.join(DATA_ROOT, d) for d in ("crops", "sessions.csv", "masks"))

from src.train_teacher import check_split
table = pd.read_csv(SPLIT)
present = {p.relative_to(CROPS).with_suffix("").as_posix() for p in Path(CROPS).rglob("*.jpg")}
table["present"] = table["path"].str.replace(r"\.[^.]+$", "", regex=True).isin(present)
print("split counts:", check_split(SPLIT), "| masks/ present:", os.path.isdir(MASKS))
print(table.groupby(["split", "class"])["present"].agg(["sum", "count"]))
assert table["present"].all(), "Some images listed in sessions.csv are missing from crops/"

## 3. Smoke test, then the full evaluation

In [ ]:
MODEL = os.path.join(RUNS, "student", "pruned", "best_model.pt")
!python -m src.evaluate_xai --data {CROPS} --masks {MASKS} --split-file {SPLIT} --model {MODEL} --out /content/smoke_xai \
    --val-per-class 2 --test-per-class 2 --shap-background 8 --shap-samples 10 --sanity-images 1 --random-repeats 1

In [ ]:
!python -m src.evaluate_xai --data {CROPS} --masks {MASKS} --split-file {SPLIT} --model {MODEL} --out {RUNS}/xai
import json
xs = json.load(open(os.path.join(RUNS, "xai", "xai_summary.json")))
print("Grad-CAM layer:", xs["gradcam_layer"], xs["layer_selection"])
display(pd.DataFrame(xs["by_method"]).T)
display(pd.DataFrame(xs["by_method_and_predicted_class"]).T)
display(pd.DataFrame(xs["sanity_mean_spearman"]))

In [ ]:
# Example maps for the paper figure (first 8 test-sample images)
import numpy as np, matplotlib.pyplot as plt
from PIL import Image
ex = np.load(os.path.join(RUNS, "xai", "example_maps.npz"))
methods = [m for m in ("gradcam", "occlusion", "shap") if f"0_{m}" in ex]
fig, axes = plt.subplots(8, 1 + len(methods), figsize=(2.4 * (1 + len(methods)), 19))
for i in range(8):
    img = Image.open(os.path.join(CROPS, str(ex[f"{i}_path"]))).convert("RGB").resize((224, 224))
    axes[i, 0].imshow(img); axes[i, 0].set_title(str(ex[f"{i}_path"]).split("/")[-1], fontsize=7)
    for j, m in enumerate(methods, 1):
        a = np.asarray(Image.fromarray(ex[f"{i}_{m}"].astype(np.float32)).resize((224, 224), Image.BILINEAR))
        axes[i, j].imshow(img); axes[i, j].imshow(a, cmap="jet", alpha=0.45)
        axes[i, j].contour(ex[f"{i}_mask"], levels=[0.5], colors="white", linewidths=0.6)
        axes[i, j].set_title(m, fontsize=8)
for ax in axes.ravel():
    ax.axis("off")
plt.tight_layout(); plt.savefig(os.path.join(RUNS, "xai", "xai_examples.png"), dpi=200); plt.show()

In [ ]:
# Bundle all result files (JSON/CSV, no weights) into one zip to share: runs/results_bundle.zip on Drive
!python -m src.collect_results --runs {RUNS} --out {RUNS}/results_bundle.zip